# Web Scrapping: Selenium

## 1. Libraries

In [ ]:
import pandas as pd
import time

# Herramientas de Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException


## 2.  Configuración e Inicialización del Navegador

Aquí configuramos y lanzamos el navegador Chrome que será controlado por nuestro script. Dejamos que el Selenium Manager integrado se encargue de gestionar el chromedriver por nosotros, lo que simplifica mucho la configuración.

In [ ]:
# Configuramos las opciones de Chrome
chrome_options = Options()
# chrome_options.add_argument("--headless")
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--lang=en-US")

# Iniciar el WebDriver de Chrome
driver = webdriver.Chrome()
# pasando nuestras 'chrome_options' como argumento.

print("WebDriver iniciado con éxito.")

## 3. Navegar a la Página de IMdb

Navegamos a la URL del Top 250 de IMDb. El paso más importante aquí es usar WebDriverWait. Le decimos a Selenium que espere hasta 10 segundos a que la lista de películas sea visible en la página antes de intentar hacer cualquier cosa. Esto hace nuestro script robusto frente a conexiones lentas.

In [ ]:
# URL del Top 250 de IMDb
driver = webdriver.Chrome()
url = "https://www.imdb.com/chart/top/"

# Pista: El objeto 'driver' tiene un método para abrir una URL. ¿Cuál es?
driver.get( url )

# Lista para guardar los datos de cada película
movies_data = []

# Selector CSS para la lista que contiene todas las películas
movie_list_selector = "ul.ipc-metadata-list"

try:
    print("Esperando a que la lista de películas cargue...")
    
    # Completa la espera para que el script se detenga hasta que la lista de películas sea visible.
    # Pista: Usa EC.visibility_of_element_located() y pásale una tupla con el método de búsqueda (By) y el selector.
    WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located(( By.XPATH, '/html/body/div[2]/main/div/div[3]/section/div/div[2]/div/ul/li[1]/div/div/div'))
    )
    print("Lista de películas encontrada. ¡A scrapear!")

except TimeoutException:
    print("Error: La lista de películas no cargó a tiempo. El script se detendrá.")
    driver.quit()

## 4. Bucle Principal de Scraping

Este es el núcleo de nuestro scraper.

- Localizamos todos los elementos <*li> que contienen la información de cada película.
- Iteramos sobre los primeros 50 elementos de esa lista.
- Dentro de cada <*li>, buscamos los datos específicos (rango, título, año, calificación y URL) usando selectores CSS.
- Usamos bloques try-except para cada atributo. Si un dato no se encuentra en una película, el script registrará "No disponible" y continuará, en lugar de detenerse por un error.

In [ ]:
# Selector para cada item (película) en la lista
movie_item_selector = 'li.ipc-metadata-list-summary-item'
#li.ipc-metadata-list-summary-item

# Pista: Usa el método 'find_elements' del driver para obtener una lista de todos los elementos que coincidan con 'movie_item_selector'.
movie_elements = driver.find_elements( By.CSS_SELECTOR, movie_item_selector )

# Iteramos solo sobre las primeras 50 películas
for movie in movie_elements[:50]:
    try:
        # --- Rango y Título ---
        # Pista: Primero, encuentra el elemento h3 con la clase 'ipc-title__text'. Luego, obtén su '.text'.
        title_element = movie.find_element(By.CSS_SELECTOR, "h3.ipc-title__text")
        full_title_text = movie.find_element(By.CSS_SELECTOR, "h3.ipc-title__text").text
        rank, title = full_title_text.split('. ', 1)

        # --- Año --- 
        year = movie.find_element(By.CSS_SELECTOR, "div.cli-title-metadata > span").text
        
        # --- Calificación --- 
        rating_element = movie.find_element(By.CSS_SELECTOR, "span.ipc-rating-star")
        rating = rating_element.text.split('\n')[0]

        # --- URL de la película ---
        # Pista: El enlace está en el atributo 'href' de la etiqueta <a>. Usa '.get_attribute()'
        url_element = movie.find_element(By.CSS_SELECTOR, "a.ipc-title-link-wrapper")
        movie_url = url_element.get_attribute('href')

        # Asegúrate de que los nombres de las variables coincidan con las que creaste arriba.
        movies_data.append({
            "Rango": rank,
            "Titulo": title_element,
            "Año": year,
            "Calificacion_IMDb": rating,
            "URL": movie_url
        })
        print(f" Scraped: #{rank} {title}")

    except Exception as e:
        print(f" Error extrayendo datos de una película. Error: {e}")
        continue

print(f"\nScraping completado. Se extrajeron datos de {len(movies_data)} películas.")

## 5. Crear el DataFrame y Guardar los Datos

Una vez que tenemos nuestra lista de diccionarios, la convertimos en un DataFrame de pandas, que es una estructura tipo tabla ideal para el análisis y almacenamiento de datos. Finalmente, lo guardamos en un archivo CSV y cerramos el navegador para liberar los recursos del sistema.

In [ ]:
if movies_data:
    # Pista: Llama a pd.DataFrame() y pásale la lista que contiene todos nuestros datos.
    df = pd.DataFrame(movies_data)
    # Pista: Usa el método '.to_csv()'. Dale un nombre de archivo, por ejemplo, "imdb_top_50.csv", y no te olvides de poner index=False.
    # [...COMPLETA AQUÍ...]
    
    df.to_csv("imbd_top_50_peliculas.csv", index=False, encoding='utf-8-sig')
    print("\n Datos guardados exitosamente.")
    display(df.head())
else:
    print("\nNo se pudo extraer ningún dato de las películas.")

# Cerrar el navegador
# Pista: Hay un método en el objeto 'driver' para cerrar todas las ventanas y terminar la sesión.
# [...COMPLETA AQUÍ...]
driver.quit()
print("\nNavegador cerrado correctamente. ¡Ejercicio terminado!")
